# Конспект 3. Рекурсия, декораторы, генераторы

- **рекурсия** - функция решает задачу, вызывая саму себя для задачи поменьше;
- **декоратор** - функция, которая берёт вашу функцию и возвращает улучшенную,
  не меняя её код;
- **генератор** - функция, которая выдаёт значения по одному, по запросу,
  а не готовым списком.

Здесь только то, что понадобится в практике №3 и в том же порядке. Подробнее, и с
дополнительными примерами смотрите материал в лекции. Читайте и **запускайте каждую ячейку**.

## Данные конспекта

In [1]:
TEXTS = [
    "Учёные открыли новый вид бактерий в глубинах Тихого океана",
    "Сборная выиграла матч со счётом 3:1",
    "Новый вирус обнаружен учёными в лаборатории Новосибирска",
]

LABELS = ["наука", "спорт", "наука"]
SCORES = [5, 3, 4]

print("записей:", len(TEXTS))

записей: 3


---

# Часть I. Рекурсия

# 1. Базовый случай и шаг

**Рекурсия** - функция вызывает саму себя. Чтобы это не продолжалось вечно,
в ней всегда две части:

- **базовый случай** - когда ответ известен сразу и вызывать себя не нужно;
- **шаг** - вызов себя для такой же задачи, но **поменьше**.

Каждый шаг приближает к базовому случаю, где заканчивается выполнение функции.

In [2]:
def countdown(n):
    if n <= 0:              # базовый случай: считать дальше некуда
        print("старт!")
        return
    print(n)
    countdown(n - 1)        # шаг: та же задача, но на единицу меньше

countdown(3)

3
2
1
старт!


Без базового случая функция вызывала бы себя, пока Python не остановит её
ошибкой `RecursionError: maximum recursion depth exceeded`.

In [3]:
def forever(n):
    forever(n - 1)          # базового случая нет

# раскомментируйте и посмотрите на последнюю строку ошибки:
# forever(3)

---

# 2. Рекурсия возвращает результат

Чаще всего рекурсивная функция не печатает, а **возвращает** результат. Рецепт для списка:
ответ = **первый элемент** + **ответ для остальных**. Базовый случай - пустой
список: для него ответ известен сразу.

In [4]:
def total(values):
    if len(values) == 0:                    # базовый случай
        return 0
    return values[0] + total(values[1:])    # первый + сумма остальных

print(total(SCORES))         # 5 + (3 + (4 + 0)) = 12

12


Как это выполняется: `total([5, 3, 4])` не может ответить решить сразу и ждёт
`total([3, 4])`, тот ждёт `total([4])` и `total([])`. Пустой список отвечает `0`,
и ответы возвращаются обратно по цепочке: `4`, потом `7`, потом `12`.

> Сумму проще посчитать циклом или `sum`, и в реальном коде так и делают.
> Рекурсия нужна там, где данные сами вложены друг в друга: папки в папках,
> рубрики в рубриках, JSON. Здесь мы разбираемся с самим механизмом.

---

# 3. Цена рекурсии: Фибоначчи

Числа Фибоначчи: `0, 1, 1, 2, 3, 5, 8, ...` , здесь каждое число равно сумме двух предыдущих.
Исходя из определения, применение рекурсии само просится для применения.
Посчитаем, сколько раз функция вызывается.

Счётчик храним в **словаре**, функция меняет содержимое словаря, а не присваивает
новое значение имени, поэтому изменение видно снаружи.

In [5]:
CALLS = {"n": 0}

def fib(n):
    CALLS["n"] += 1
    if n < 2:                          # базовый случай: fib(0) = 0, fib(1) = 1
        return n
    return fib(n - 1) + fib(n - 2)     # шаг: два вызова себя

for n in [5, 10, 15, 20]:
    CALLS["n"] = 0
    print(f"fib({n}) = {fib(n)}, вызовов: {CALLS['n']}")

fib(5) = 5, вызовов: 15
fib(10) = 55, вызовов: 177
fib(15) = 610, вызовов: 1973
fib(20) = 6765, вызовов: 21891


Каждый вызов порождает **два** новых, и одни и те же значения считаются снова
и снова: `fib(3)` внутри `fib(5)` считается дважды, а `fib(2)` трижды. Число вызовов
растёт примерно в 11 раз на каждые пять шагов.

---

# Часть II. Декораторы

# 4. Функция - это значение

Имя функции **без скобок** - это сама функция. Её можно хранить в переменной,
передать в другую функцию и вернуть из функции. На этом держатся декораторы.

In [6]:
def word_count(text):
    return len(text.split())

counter = word_count             # без скобок: не вызов, а сама функция
print(counter(TEXTS[1]))

def apply_to_all(func, items):
    return [func(item) for item in items]

print(apply_to_all(word_count, TEXTS))

6
[9, 6, 7]


---

# 5. Первый декоратор

**Декоратор** - это функция, которая принимает функцию и возвращает **новую**.
Новая функция (её принято называть `wrapper`, «обёртка») вызывает исходную
и делает что-то до или после.

Запись `@loud` над `def`, аналогична записи `double = loud(double)` после него.

In [7]:
def loud(func):
    def wrapper(*args, **kwargs):
        print("  до вызова", func.__name__)
        result = func(*args, **kwargs)
        print("  после вызова, результат:", result)
        return result
    return wrapper

@loud
def double(x):
    return x * 2

print(double(5))

  до вызова double
  после вызова, результат: 10
10


Три строки в этом шаблоне обязательны и их забывают чаще всего:

- `*args, **kwargs` у `wrapper` - нужны, чтобы обёртка подходила к функции с любыми
  параметрами;
- `func(*args, **kwargs)` - передать все аргументы дальше исходной функции;
- `return result` - без него обёрнутая функция всегда вернёт `None`.

> `wrapper` помнит `func`, хотя `loud` уже закончила работу. Такая внутренняя функция
> называется **замыканием**.

---

# 6. Декоратор с памятью: счётчик вызовов

У функции, как у любого объекта, можно создать **атрибут** - переменную,
прикреплённую к ней: `wrapper.calls = 0`. Обёртка увеличивает его при каждом
вызове, а прочитать его можно снаружи: `double.calls`.

In [8]:
def count_calls(func):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def triple(x):
    return x * 3

for x in range(4):
    triple(x)
print("triple вызвана", triple.calls, "раза")

triple вызвана 4 раза


Декоратор можно применить и на готовую функцию без `@`: `word_count = count_calls(word_count)`.
После этого **все**, кто вызывает `word_count` по имени, в том числе другие ваши
функции - вызывают счётную версию.

---

# 7. Кэш: Фибоначчи за 49 вызовов

**Кэш** - это память ответов. Если функцию уже вызывали с такими аргументами, обёртка
возвращает запомненный ответ и не считает заново. Ключ словаря - это кортеж
аргументов `args`, значение - результат.

In [9]:
def cache(func):
    def wrapper(*args):
        if args not in wrapper.memory:          # такого ещё не считали
            wrapper.memory[args] = func(*args)  # считаем и запоминаем
        return wrapper.memory[args]             # отдаём из памяти
    wrapper.memory = {}
    return wrapper

@count_calls
@cache
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

print("fib_fast(20) =", fib_fast(20), " вызовов:", fib_fast.calls)

fib_fast(20) = 6765  вызовов: 39


Код Фибоначчи тот же самый, изменилась одна строка `@cache`. Вместо 21 891 вызова -
39: каждое значение считается ровно один раз.

> Декораторов может быть несколько: применяются они **снизу вверх**. Здесь сначала
> `cache`, потом `count_calls` поверх него, поэтому счётчик видит все обращения,
> включая те, на которые кэш ответил из памяти.

---

# Часть III. Генераторы

# 8. `yield` и `next`

Если в функции есть `yield`, это **генераторная функция**. Её вызов **не выполняет
тело**, а создаёт объект-генератор. Значения из него берут встроенной функцией
`next()`: каждый `next` выполняет тело до следующего `yield`, отдаёт значение
и **замирает** на этом месте до следующего запроса.

In [10]:
def three_numbers():
    yield 1
    yield 2
    yield 3

gen = three_numbers()
print(gen)                  # не числа, а объект-генератор
print(next(gen))
print(next(gen))
print(next(gen))
# print(next(gen))          # раскомментируйте: значения кончились - StopIteration

<generator object three_numbers at 0x1072a62c0>
1
2
3


Обычно генератор перебирают циклом `for`: он сам вызывает `next` и сам
останавливается, когда значения кончились.

In [11]:
for number in three_numbers():
    print(number)

1
2
3


---

# 9. Ленивость: значения по запросу

Генератор **ленивый**: он не считает ничего заранее. Это видно, если добавить
`print` внутрь, и сообщения появляются только в момент запроса.

In [12]:
def texts_loud(texts):
    for i, text in enumerate(texts):
        print(f"  генератор: выдаю запись {i}")
        yield text

gen = texts_loud(TEXTS)
print("генератор создан, но ещё ничего не выдал")
print("получил:", next(gen))
print("делаю что-то ещё...")
print("получил:", next(gen))

генератор создан, но ещё ничего не выдал
  генератор: выдаю запись 0
получил: Учёные открыли новый вид бактерий в глубинах Тихого океана
делаю что-то ещё...
  генератор: выдаю запись 1
получил: Сборная выиграла матч со счётом 3:1


> Поэтому генераторы используют для больших данных: в памяти одновременно
> находится **одна** запись, а не весь объем данных. Миллион текстов или десять -
> генератору всё равно.

---

# 10. Генераторное выражение

Короткий генератор записывают одной строкой, как списковое выражение,
но в **круглых** скобках. Если выражение, является единственным аргументом функции,
вторые скобки не пишут: `sum(x for x in xs)`.

Главная ловушка: генератор - это **поток**, а не хранилище. Прочитанное
не вернуть, второй проход будет уже пустым.

In [13]:
lengths = (len(t.split()) for t in TEXTS)    # генератор
print("первая сумма:", sum(lengths))
print("вторая сумма:", sum(lengths))          # 0: генератор уже прочитан

print("длинных записей:", sum(1 for t in TEXTS if len(t.split()) > 7))

values = [len(t.split()) for t in TEXTS]      # нужен дважды - делаем список
print(sum(values), max(values))

первая сумма: 22
вторая сумма: 0
длинных записей: 1
22 9


---

# 11. Цепочка генераторов

Генераторы соединяют в цепочку: каждый берёт значения у предыдущего и передаёт
дальше. Значения проходят по цепочке **по одному**, а если нужное найдено раньше,
остальное даже не прочитается.

In [14]:
def read(texts):
    for text in texts:
        print(f"  читаю: {text[:25]}...")
        yield text

def words(stream):
    for text in stream:
        for word in text.lower().split():
            yield word

for word in words(read(TEXTS)):
    print(word)
    if word == "выиграла":
        break               # третья запись так и не прочитана

  читаю: Учёные открыли новый вид ...
учёные
открыли
новый
вид
бактерий
в
глубинах
тихого
океана
  читаю: Сборная выиграла матч со ...
сборная
выиграла


---

# Шпаргалка занятия

| Что нужно | Как пишется | Замечание |
| --- | --- | --- |
| рекурсия | базовый случай + вызов себя для задачи поменьше | без базы - `RecursionError` |
| рекурсия над списком | `values[0] + total(values[1:])` | база: пустой список |
| счётчик снаружи функции | `CALLS = {"n": 0}`, внутри `CALLS["n"] += 1` | меняем словарь |
| функция как значение | `counter = word_count` (без скобок) | можно передать в функцию |
| декоратор | `def deco(func): def wrapper(*args, **kwargs): ...; return wrapper` | |
| внутри обёртки | `result = func(*args, **kwargs)`, `return result` | не забыть `return` |
| применить | `@deco` над `def` или `f = deco(f)` | одно и то же |
| атрибут функции | `wrapper.calls = 0`, снаружи `f.calls` | память декоратора |
| кэш | словарь `args → результат` в `wrapper.memory` | |
| несколько декораторов | применяются снизу вверх | |
| генератор | функция с `yield` | вызов только создаёт объект |
| следующее значение | `next(gen)` | в конце - `StopIteration` |
| перебрать | `for x in gen:` | сам останавливается |
| генераторное выражение | `(f(x) for x in xs)`, `sum(f(x) for x in xs)` | круглые скобки |
| читается один раз | нужен повторно - `list(...)` | второй проход пуст |